In [1]:
#!/usr/bin/env python3
#"""Constrained model predictive control of a Cho et al. thrust AMB.

#Reference
#---------
#Y. M. Cho, S. Srinivasan, J.-H. Oh and H. S. Kim, "Modelling and system
#identification of active magnetic bearing systems", Mathematical and Computer
#Modelling of Dynamical Systems, 13(2), 125-142, 2007.

#Cho et al. give the bias-linearized actuator and PWM transconductance amplifier
#in their Eqs. (13)-(14). With the user-requested reduced axial model, the plant
#implemented here is exactly

#                  k_i K_amp
# G(s)=X/V_sp = --------------------------- .
#                (m s^2-k_x)(tau s+1)

#The state realization is
#    x_dot = v
#    m v_dot = k_x*x + k_i*i_c + F_disturbance
#    tau i_c_dot = -i_c + K_amp*V_sp.

#IMPORTANT SIGN CONVENTION: ``k_x`` below is the positive magnitude of the
#destabilizing (negative mechanical stiffness) term in ``m*x_ddot-k_x*x``.
#Cho et al. describe their actuator coefficient k_x as signed and negative.
#Convert an experimentally reported signed coefficient to the convention used
#here before entering it.

#The script contains an automatic RK4-versus-analytical-transfer-function test,
#a noisy filtered sensor, digital delay, PWM/voltage/current limits, anti-windup,
#a constrained digital optimizer, explicit disturbance injection, plots and
#validation metrics. It is a small-signal bias-linearized model; x is axial
#displacement from the nominal gap, not the absolute gap itself.

#Dependencies: numpy, scipy, matplotlib
#Run: python Cho2007_Model_Predictive_Control_Thrust_AMB.py
#"""
from __future__ import annotations

import argparse
import json
from collections import deque
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Deque, Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
from scipy import linalg, optimize, signal


@dataclass
class AMBPlantParameters:
#    """All replaceable physical and implementation parameters in SI units."""

    # Bias-linearized axial plant: m*x_ddot-kx*x=ki*ic+F_d.
    rotor_mass: float = 0.42                    # m [kg]
    negative_stiffness_magnitude: float = 120.0 # k_x [N/m], enter POSITIVE magnitude
    current_force_gain: float = 3.8             # k_i [N/A]
    amplifier_gain: float = 0.45                # K_amp [A/V]
    amplifier_time_constant: float = 0.006      # tau [s]
    nominal_air_gap: float = 0.014              # [m]
    bias_current: float = 2.20                   # physical DC bias [A]

    # Constraints. ic is incremental control current about bias_current.
    control_current_min: float = -1.80           # [A]
    control_current_max: float = 1.80            # [A]
    voltage_setpoint_limit: float = 10.0         # |V_sp| [V]
    dc_bus_voltage: float = 48.0                 # PWM DC link [V]
    pwm_duty_limit: float = 0.95                 # absolute duty ratio
    amplifier_efficiency: float = 0.97

    # Position safety region expressed about nominal_air_gap.
    minimum_air_gap: float = 0.010               # [m]
    maximum_air_gap: float = 0.018               # [m]
    initial_displacement: float = 0.25e-3        # x(0) [m]
    initial_velocity: float = 0.0                # [m/s]

    def effective_voltage_limit(self) -> float:
        return min(self.voltage_setpoint_limit,
                   self.dc_bus_voltage*self.pwm_duty_limit)

    def analytical_tf(self) -> Tuple[np.ndarray, np.ndarray]:
#        """Numerator/denominator of the exact requested G(s)."""
        m, kx, tau = (self.rotor_mass, self.negative_stiffness_magnitude,
                      self.amplifier_time_constant)
        numerator = np.array([self.current_force_gain*self.amplifier_gain])
        # (m*s^2-kx)*(tau*s+1)
        denominator = np.array([m*tau, m, -kx*tau, -kx])
        return numerator, denominator


@dataclass
class SensorParameters:
    sensitivity: float = 800.0       # [V/m]
    bias_voltage: float = 0.0008     # residual uncompensated offset [V]
    noise_variance: float = 4.0e-7   # voltage-noise variance [V^2]
    lowpass_cutoff_hz: float = 100.0 # below Nyquist for the default 2 ms MPC rate


@dataclass
@dataclass
@dataclass
class MPCParameters:
#    """Prediction, cost, observer and constraint configuration."""
    sample_time: float = 0.002
    prediction_horizon: int = 60
    control_horizon: int = 10
    computational_delay_samples: int = 0
    # Quadratic stage weights in physical SI coordinates.
    position_weight: float = 2.0e10
    velocity_weight: float = 5.0e6
    current_weight: float = 0.05
    voltage_weight: float = 0.001
    voltage_rate_weight: float = 5.0
    terminal_position_multiplier: float = 20.0
    soft_gap_penalty: float = 1.0e12
    maximum_soft_gap_violation: float = 10.0e-3
    maximum_voltage_change_per_sample: float = 2.5
    # Steady-state Kalman disturbance observer tuning.
    observer_state_noise: Tuple[float,float,float,float] = (1e-10,1e-6,2e-4,1e-3)
    observer_measurement_noise: float = 2.0e-10
    solver_max_iterations: int = 80
    solver_tolerance: float = 1e-7


class ThrustAMBPlant:
#    """Saturated RK4 realization of the requested third-order transfer model."""

    def __init__(self, p: AMBPlantParameters):
        self.p = p
        self.state = np.array([p.initial_displacement, p.initial_velocity, 0.0])
        self.voltage_applied = 0.0
        self.duty_ratio = 0.0

    def _limited_voltage(self, voltage_demand: float) -> float:
        p = self.p
        voltage = float(np.clip(voltage_demand, -p.effective_voltage_limit(),
                                p.effective_voltage_limit()))
        self.duty_ratio = voltage/p.dc_bus_voltage
        return p.amplifier_efficiency*voltage

    def derivative(self, state: np.ndarray, voltage_demand: float,
                   disturbance_force: float, apply_limits: bool = True) -> np.ndarray:
        p = self.p
        x, velocity, incremental_current = state
        voltage = self._limited_voltage(voltage_demand) if apply_limits else voltage_demand
        current_target = p.amplifier_gain*voltage
        if apply_limits:
            current_target = float(np.clip(current_target,p.control_current_min,
                                           p.control_current_max))
        current_dot = (current_target-incremental_current)/p.amplifier_time_constant
        # Stop integration from driving an already-saturated current farther out.
        if apply_limits and ((incremental_current <= p.control_current_min and current_dot < 0)
                             or (incremental_current >= p.control_current_max and current_dot > 0)):
            current_dot = 0.0
        acceleration = (p.negative_stiffness_magnitude*x
                        +p.current_force_gain*incremental_current
                        +disturbance_force)/p.rotor_mass
        return np.array([velocity,acceleration,current_dot])

    def step(self, voltage_demand: float, disturbance_force: float,
             dt: float, substeps: int = 4) -> None:
#        """Classical fixed-step RK4 within one digital-control interval."""
        h = dt/substeps
        for _ in range(substeps):
            s = self.state
            k1 = self.derivative(s,voltage_demand,disturbance_force)
            k2 = self.derivative(s+0.5*h*k1,voltage_demand,disturbance_force)
            k3 = self.derivative(s+0.5*h*k2,voltage_demand,disturbance_force)
            k4 = self.derivative(s+h*k3,voltage_demand,disturbance_force)
            self.state = s+h*(k1+2*k2+2*k3+k4)/6.0
        self.state[2] = np.clip(self.state[2],self.p.control_current_min,
                                self.p.control_current_max)
        self.voltage_applied = self._limited_voltage(voltage_demand)


class SensorFilter:
#    """Sensitivity, residual bias, Gaussian noise and Butterworth filtering."""

    def __init__(self, p: SensorParameters, sample_time: float,
                 initial_displacement: float, seed: int = 21):
        self.p = p
        self.rng = np.random.default_rng(seed)
        k = np.tan(np.pi*p.lowpass_cutoff_hz*sample_time)
        norm = 1.0/(1.0+np.sqrt(2.0)*k+k*k)
        self.b = np.array([k*k*norm,2*k*k*norm,k*k*norm])
        self.a = np.array([1.0,2*(k*k-1)*norm,
                           (1-np.sqrt(2.0)*k+k*k)*norm])
        initial_v = p.sensitivity*initial_displacement+p.bias_voltage
        self.input_history = np.array([initial_v,initial_v])
        self.output_history = np.array([initial_v,initial_v])

    def measure(self, displacement: float) -> float:
        sensor_voltage = (self.p.sensitivity*displacement+self.p.bias_voltage
                          +self.rng.normal(0.0,np.sqrt(self.p.noise_variance)))
        y = (self.b[0]*sensor_voltage+self.b[1]*self.input_history[0]
             +self.b[2]*self.input_history[1]-self.a[1]*self.output_history[0]
             -self.a[2]*self.output_history[1])
        self.input_history[1],self.input_history[0] = self.input_history[0],sensor_voltage
        self.output_history[1],self.output_history[0] = self.output_history[0],y
        # Divide by sensitivity; residual bias remains visible to the controller.
        return y/self.p.sensitivity


class ModelPredictiveController:
#    """Offset-free constrained MPC with a condensed quadratic program.

#    The internal Cho-model state is [x, x_dot, i_c]. A constant unknown force
#    state d is appended for offset-free disturbance estimation. The discrete
#    observer model is obtained by exact zero-order-hold discretization.

#    To penalize and constrain voltage rate, the predictor is augmented by the
#    previously applied voltage: z=[x,x_dot,i_c,d,u_previous]. Its dynamics are

#        z(k+1)=Abar*z(k)+Bbar*Delta_u(k).

#    Repeated substitution gives the condensed prediction Z=F*z0+Phi*Delta_U.
#    The QP cost is

#        J = sum ||z-r||_Q^2 + sum Rdu*Delta_u^2 + rho*slack^2,

#    subject to hard voltage/current/rate constraints and soft gap constraints.
#    Future Delta_u is zero after Nc, so Np may be much longer than Nc.
#    """
    def __init__(self,mp: MPCParameters,pp: AMBPlantParameters):
        self.m,self.p=mp,pp
        self._build_discrete_observer()
        self._build_prediction_matrices()
        self.xhat=np.zeros(4)
        self.first_measurement=True
        self.previous_voltage=0.0
        self.warm=np.zeros(mp.control_horizon+1)
        self.last_prediction=np.zeros(mp.prediction_horizon)

    def _build_discrete_observer(self):
        p,m=self.p,self.m
        A3=np.array([[0,1,0],[p.negative_stiffness_magnitude/p.rotor_mass,0,
                    p.current_force_gain/p.rotor_mass],[0,0,-1/p.amplifier_time_constant]])
        B3=np.array([[0],[0],[p.amplifier_gain/p.amplifier_time_constant]])
        E3=np.array([[0],[1/p.rotor_mass],[0]])
        A4=np.zeros((4,4)); A4[:3,:3]=A3; A4[:3,3:4]=E3
        B4=np.vstack([B3,[[0.0]]]); C4=np.array([[1.,0.,0.,0.]])
        self.Ao,self.Bo,self.Co,_,_=signal.cont2discrete(
            (A4,B4,C4,np.zeros((1,1))),m.sample_time,method='zoh')
        Qn=np.diag(m.observer_state_noise); Rn=np.array([[m.observer_measurement_noise]])
        covariance=np.asarray(np.real_if_close(
            linalg.solve_discrete_are(self.Ao.T,self.Co.T,Qn,Rn)))
        self.L=covariance@self.Co.T@np.linalg.inv(self.Co@covariance@self.Co.T+Rn)

    def _build_prediction_matrices(self):
        m=self.m; nx=5; Np,Nc=m.prediction_horizon,m.control_horizon
        self.Abar=np.zeros((nx,nx)); self.Abar[:4,:4]=self.Ao
        self.Abar[:4,4:5]=self.Bo; self.Abar[4,4]=1.0
        self.Bbar=np.vstack([self.Bo,[[1.0]]])
        F=np.zeros((Np*nx,nx)); Phi=np.zeros((Np*nx,Nc))
        Fk=np.eye(nx); Phik=np.zeros((nx,Nc))
        for k in range(Np):
            Fk=self.Abar@Fk; Phik=self.Abar@Phik
            if k<Nc: Phik[:,k:k+1]+=self.Bbar
            F[k*nx:(k+1)*nx]=Fk; Phi[k*nx:(k+1)*nx]=Phik
        self.F,self.Phi=F,Phi
        q=np.array([m.position_weight,m.velocity_weight,m.current_weight,0.,m.voltage_weight])
        Qbar=np.kron(np.eye(Np),np.diag(q))
        Qbar[-nx,-nx]*=m.terminal_position_multiplier
        self.Qbar=Qbar
        self.Hdu=2*(Phi.T@Qbar@Phi+m.voltage_rate_weight*np.eye(Nc))

        selectors=lambda index: np.kron(np.eye(Np),np.eye(1,nx,index))
        self.Sx,self.Si,self.Su=selectors(0),selectors(2),selectors(4)
        self.Fx,self.Phix=self.Sx@F,self.Sx@Phi
        self.Fi,self.Phii=self.Si@F,self.Si@Phi
        self.Fu,self.Phiu=self.Su@F,self.Su@Phi

    def _observe(self,measurement: float,applied_voltage: float):
        if self.first_measurement:
            self.xhat[0]=measurement; self.first_measurement=False
        prediction=self.Ao@self.xhat+self.Bo.ravel()*applied_voltage
        innovation=measurement-(self.Co@prediction).item()
        self.xhat=prediction+self.L.ravel()*innovation

    def compute(self,reference: float,measurement: float,applied_voltage: float):
        import time
        self._observe(measurement,applied_voltage)
        m,p=self.m,self.p; Np,Nc=m.prediction_horizon,m.control_horizon
        z0=np.r_[self.xhat,self.previous_voltage]
        free=self.F@z0
        target=np.zeros_like(free); target[0::5]=reference
        fdu=2*self.Phi.T@self.Qbar@(free-target)

        # Decision q=[Delta_u(0:Nc-1), gap_slack].
        H=np.zeros((Nc+1,Nc+1)); H[:Nc,:Nc]=self.Hdu
        # sigma is dimensionless; physical slack = sigma*maximum_soft_gap_violation.
        H[-1,-1]=2*m.soft_gap_penalty*m.maximum_soft_gap_violation**2
        f=np.r_[fdu,0.0]
        xfree=self.Fx@z0; ifree=self.Fi@z0; ufree=self.Fu@z0
        xmin=p.minimum_air_gap-p.nominal_air_gap
        xmax=p.maximum_air_gap-p.nominal_air_gap
        zero_slack=np.zeros((Np,1))
        minus_slack=-m.maximum_soft_gap_violation*np.ones((Np,1))
        Aineq=np.vstack([
            np.hstack([self.Phiu,zero_slack]),np.hstack([-self.Phiu,zero_slack]),
            np.hstack([self.Phii,zero_slack]),np.hstack([-self.Phii,zero_slack]),
            np.hstack([self.Phix,minus_slack]),np.hstack([-self.Phix,minus_slack])])
        bineq=np.r_[np.full(Np,p.effective_voltage_limit())-ufree,
                    -np.full(Np,-p.effective_voltage_limit())+ufree,
                    np.full(Np,p.control_current_max)-ifree,
                    -np.full(Np,p.control_current_min)+ifree,
                    np.full(Np,xmax)-xfree,-np.full(Np,xmin)+xfree]
        # Row scaling is essential: raw constraints mix V, A and m and are
        # badly conditioned for a generic SQP solver.
        row_scale=np.r_[np.full(2*Np,p.effective_voltage_limit()),
                        np.full(2*Np,max(abs(p.control_current_min),p.control_current_max)),
                        np.full(2*Np,0.5*(p.maximum_air_gap-p.minimum_air_gap))]
        Aineq=Aineq/row_scale[:,None]; bineq=bineq/row_scale
        constraint=optimize.LinearConstraint(Aineq,-np.inf,bineq)
        bounds=optimize.Bounds(
            np.r_[np.full(Nc,-m.maximum_voltage_change_per_sample),0.0],
            np.r_[np.full(Nc,m.maximum_voltage_change_per_sample),1.0])
        objective_scale=max(float(np.max(np.diag(H))),1.0)
        Hs,fs=H/objective_scale,f/objective_scale
        objective=lambda q: 0.5*q@Hs@q+fs@q
        gradient=lambda q: Hs@q+fs
        start=time.perf_counter()
        result=optimize.minimize(objective,self.warm,jac=gradient,method='SLSQP',
            bounds=bounds,constraints=[constraint],options={'maxiter':m.solver_max_iterations,
            'ftol':m.solver_tolerance,'disp':False})
        elapsed=1e3*(time.perf_counter()-start)
        if result.success and np.all(np.isfinite(result.x)):
            solution=result.x
        else:
            # Stabilizing observer-feedback fallback; failure remains logged.
            fallback=(-600.0*self.xhat[0]-25.0*self.xhat[1]
                      -self.xhat[3]/(p.current_force_gain*p.amplifier_gain))
            fallback=float(np.clip(fallback,-p.effective_voltage_limit(),p.effective_voltage_limit()))
            solution=np.zeros(Nc+1)
            solution[0]=np.clip(fallback-self.previous_voltage,
                                -m.maximum_voltage_change_per_sample,
                                m.maximum_voltage_change_per_sample)
        du=float(solution[0])
        voltage=float(np.clip(self.previous_voltage+du,-p.effective_voltage_limit(),
                              p.effective_voltage_limit()))
        self.previous_voltage=voltage
        shifted=np.r_[solution[1:Nc],solution[Nc-1],solution[-1]]
        self.warm=shifted
        self.last_prediction=(self.Fx@z0+self.Phix@solution[:Nc]).ravel()
        physical_slack=float(solution[-1]*m.maximum_soft_gap_violation)
        return voltage,reference-measurement,elapsed,physical_slack,int(result.success),int(result.nit)


class SimulationEnvironment:
    def __init__(self,plant: ThrustAMBPlant,sensor: SensorFilter,
                 controller: ModelPredictiveController,duration: float=3.0):
        self.plant,self.sensor,self.controller = plant,sensor,controller
        self.duration,self.ts = duration,controller.m.sample_time
        n=max(0,controller.m.computational_delay_samples)
        self.delay_line=deque([0.0]*(n+1),maxlen=n+1)

    @staticmethod
    def default_disturbance(t: float) -> float:
#        """Step at 0.5 s, sinusoidal surge after 1.5 s and finite pulse."""
        step=0.55 if t>=0.5 else 0.0
        sine=0.10*np.sin(2*np.pi*1.2*(t-1.5)) if t>=1.5 else 0.0
        pulse=-0.30 if 2.15<=t<2.22 else 0.0
        return step+sine+pulse

    def run(self,disturbance: Optional[Callable[[float],float]]=None) -> Dict[str,np.ndarray]:
        disturbance=disturbance or self.default_disturbance
        n=int(round(self.duration/self.ts))+1
        names=('time','displacement','air_gap','measured_displacement','velocity',
               'incremental_current','total_current','voltage_demand','voltage_applied',
               'pwm_duty','disturbance','error','solver_time_ms','gap_slack_m',
               'solver_success','solver_iterations','predicted_peak_displacement')
        log={name:np.zeros(n) for name in names}
        applied_command=0.0
        for k in range(n):
            t=k*self.ts
            x,v,ic=self.plant.state
            measured=self.sensor.measure(x)
            voltage,error,solve_ms,slack,success,iterations=self.controller.compute(
                0.0,measured,applied_command)
            delayed=self.delay_line.popleft(); self.delay_line.append(voltage)
            force=disturbance(t)
            values=(t,x,self.plant.p.nominal_air_gap+x,measured,v,ic,
                    self.plant.p.bias_current+ic,voltage,self.plant.voltage_applied,
                    self.plant.duty_ratio,force,error,solve_ms,slack,success,iterations,
                    np.max(np.abs(self.controller.last_prediction)))
            for name,value in zip(names,values): log[name][k]=value
            if k<n-1:
                self.plant.step(delayed,force,self.ts)
                applied_command=delayed
        return log


def validate_transfer_function(p: AMBPlantParameters,dt: float=1e-4,
                               duration: float=0.12) -> Dict[str,float]:
#    """Compare unsaturated RK4 state equations with scipy.signal G(s)."""
    t=np.arange(0.0,duration+0.5*dt,dt)
    u=0.08*np.sin(2*np.pi*7.0*t)+0.03*(t>=0.035)
    numerical=np.zeros_like(t); state=np.zeros(3)

    def f(s,uk):
        x,v,i=s
        return np.array([v,(p.negative_stiffness_magnitude*x+p.current_force_gain*i)/p.rotor_mass,
                         (p.amplifier_gain*uk-i)/p.amplifier_time_constant])
    for k in range(len(t)-1):
        h=dt; u0=u[k]; u1=u[k+1]; um=0.5*(u0+u1)
        k1=f(state,u0); k2=f(state+0.5*h*k1,um)
        k3=f(state+0.5*h*k2,um); k4=f(state+h*k3,u1)
        state += h*(k1+2*k2+2*k3+k4)/6.0
        numerical[k+1]=state[0]
    numerator,denominator=p.analytical_tf()
    _,analytical,_=signal.lsim(signal.TransferFunction(numerator,denominator),U=u,T=t)
    error=numerical-analytical
    scale=max(np.max(np.abs(analytical)),1e-12)
    return {'tf_validation_RMSE_m':float(np.sqrt(np.mean(error**2))),
            'tf_validation_normalized_RMSE_percent':float(100*np.sqrt(np.mean(error**2))/scale),
            'tf_validation_peak_error_m':float(np.max(np.abs(error)))}


def convergence_time(t: np.ndarray,x: np.ndarray,event: float=0.5,
                     tolerance: float=0.10e-3,dwell: float=0.15,
                     search_end: float=1.49) -> float:
    dt=t[1]-t[0]; width=max(1,int(round(dwell/dt)))
    start=int(np.searchsorted(t,event)); stop=min(int(np.searchsorted(t,search_end)),len(t)-width)
    inside=np.abs(x)<=tolerance
    for k in range(start,stop):
        if np.all(inside[k:k+width]): return float(t[k]-event)
    return float('nan')


def metrics(log: Dict[str,np.ndarray],p: AMBPlantParameters) -> Dict[str,float]:
    x=log['displacement']; gap=log['air_gap']
    return {'position_RMSE_um':float(1e6*np.sqrt(np.mean(x*x))),
            'peak_gap_deviation_um':float(1e6*np.max(np.abs(x))),
            'maximum_total_current_A':float(np.max(log['total_current'])),
            'maximum_incremental_current_A':float(np.max(np.abs(log['incremental_current']))),
            'maximum_voltage_demand_V':float(np.max(np.abs(log['voltage_demand']))),
            'convergence_time_after_step_s':convergence_time(log['time'],x),
            'minimum_air_gap_mm':float(1e3*np.min(gap)),
            'maximum_air_gap_mm':float(1e3*np.max(gap)),
            'safety_bound_violations':int(np.sum((gap<p.minimum_air_gap)|(gap>p.maximum_air_gap))),
            'mean_solver_time_ms':float(np.mean(log['solver_time_ms'])),
            'median_solver_time_ms':float(np.median(log['solver_time_ms'])),
            'solver_99th_percentile_ms':float(np.percentile(log['solver_time_ms'],99)),
            'maximum_solver_time_ms':float(np.max(log['solver_time_ms'])),
            'solves_within_sample_time_percent':float(100*np.mean(
                log['solver_time_ms']<=1e3*(log['time'][1]-log['time'][0]))),
            'solver_success_percent':float(100*np.mean(log['solver_success'])),
            'maximum_gap_slack_um':float(1e6*np.max(log['gap_slack_m']))}


def plot_results(log: Dict[str,np.ndarray],p: AMBPlantParameters,path: Path) -> None:
    t=log['time']; fig,ax=plt.subplots(4,1,figsize=(11,10),sharex=True)
    ax[0].plot(t,1e3*log['air_gap'],label='actual air gap',lw=1.5)
    ax[0].axhline(1e3*p.nominal_air_gap,color='k',ls=':',label='reference')
    ax[0].axhspan(1e3*p.minimum_air_gap,1e3*p.maximum_air_gap,color='green',alpha=.08,label='safe region')
    ax[0].set_ylabel('Air gap [mm]'); ax[0].legend(ncol=3); ax[0].grid(alpha=.25)
    ax[1].plot(t,log['total_current'],label='total coil current')
    ax[1].plot(t,log['incremental_current'],'--',label='incremental current')
    axv=ax[1].twinx(); axv.plot(t,log['voltage_demand'],color='tab:red',alpha=.7,label='Vsp demand')
    lim=p.effective_voltage_limit(); axv.axhline(lim,color='tab:red',ls=':'); axv.axhline(-lim,color='tab:red',ls=':')
    ax[1].set_ylabel('Current [A]'); axv.set_ylabel('Voltage [V]',color='tab:red')
    ax[1].legend(loc='upper left'); axv.legend(loc='upper right'); ax[1].grid(alpha=.25)
    ax[2].plot(t,log['solver_time_ms'],color='tab:green',label='QP execution time')
    ax[2].set_ylabel('Solver [ms]'); ax[2].legend(); ax[2].grid(alpha=.25)
    ax[3].plot(t,log['disturbance'],color='tab:purple',label='axial thrust disturbance')
    ax[3].set_ylabel('Load [N]'); ax[3].set_xlabel('Time [s]'); ax[3].legend(); ax[3].grid(alpha=.25)
    fig.suptitle('Cho et al. (2007) Transfer-Function AMB - Constrained MPC')
    fig.tight_layout(); fig.savefig(path,dpi=180); plt.close(fig)


def main(argv=None) -> Dict[str,float]:
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--output-dir',default='cho2007_mpc_results')
    parser.add_argument('--duration',type=float,default=3.0)
    parser.add_argument('--seed',type=int,default=21)
    args,_=parser.parse_known_args(argv) # safe inside Jupyter (-f kernel.json)
    out=Path(args.output_dir); out.mkdir(parents=True,exist_ok=True)
    pp,sp,mp=AMBPlantParameters(),SensorParameters(),MPCParameters()
    tf_check=validate_transfer_function(pp)
    plant=ThrustAMBPlant(pp); sensor=SensorFilter(sp,mp.sample_time,pp.initial_displacement,args.seed)
    controller=ModelPredictiveController(mp,pp)
    log=SimulationEnvironment(plant,sensor,controller,args.duration).run()
    report={**metrics(log,pp),**tf_check,
            'prediction_horizon':mp.prediction_horizon,'control_horizon':mp.control_horizon,
            'sample_time_s':mp.sample_time}
    plot_results(log,pp,out/'cho2007_mpc_response.png')
    np.savez_compressed(out/'simulation_data.npz',**log)
    with open(out/'metrics.json','w',encoding='utf-8') as f: json.dump(report,f,indent=2)
    with open(out/'parameters.json','w',encoding='utf-8') as f:
        json.dump({'plant':asdict(pp),'sensor':asdict(sp),'mpc':asdict(mp)},f,indent=2)
    np.savez_compressed(out/'mpc_prediction_model.npz',Ao=controller.Ao,Bo=controller.Bo,
        Co=controller.Co,L=controller.L,Abar=controller.Abar,Bbar=controller.Bbar,
        F=controller.F,Phi=controller.Phi)
    print('\nCho et al. transfer-function constrained MPC validation')
    print('-'*62)
    for key,value in report.items():
        shown='not reached' if isinstance(value,float) and np.isnan(value) else f'{value:.6g}'
        print(f'{key:43s}: {shown}')
    print(f'Results directory                          : {out.resolve()}')
    return report


if __name__=='__main__':
    main()


Cho et al. transfer-function constrained MPC validation
--------------------------------------------------------------
position_RMSE_um                           : 135.327
peak_gap_deviation_um                      : 723.32
maximum_total_current_A                    : 2.26972
maximum_incremental_current_A              : 0.339614
maximum_voltage_demand_V                   : 0.841546
convergence_time_after_step_s              : 0.238
minimum_air_gap_mm                         : 13.5099
maximum_air_gap_mm                         : 14.7233
safety_bound_violations                    : 0
mean_solver_time_ms                        : 2.15043
median_solver_time_ms                      : 1.9896
solver_99th_percentile_ms                  : 6.2987
maximum_solver_time_ms                     : 21.4504
solves_within_sample_time_percent          : 50.1666
solver_success_percent                     : 100
maximum_gap_slack_um                       : 24.0096
tf_validation_RMSE_m                       : 